# clean implementation of tool calling and gradio demo

In [14]:
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# Initialize the OpenAI client
openai_client = OpenAI()

# 1. Simple flight database dictionary
FLIGHT_PRICES = {
    "london": "$550",
    "paris": "$600",
    "new york": "$450",
    "tokyo": "$950",
    "berlin": "$500"
}

def get_ticket_prices(city: str) -> str:
    cleaned_city = city.lower().strip()
    
    print(f"\n[Tool Executing] Looking up flight prices for: '{cleaned_city}'")
    
    if cleaned_city in FLIGHT_PRICES:
        price = FLIGHT_PRICES[cleaned_city]
        return f"The ticket for {city.title()} is {price}."
    
    return f"Tickets for {city.title()} are currently not available."

def chat(user_prompt, history):
    available_functions = {
        "get_ticket_prices": get_ticket_prices
    }
    
    tools = [{
        "type": "function",
        "function": {
            "name": "get_ticket_prices",
            "description": "Get the current flight ticket price for a destination city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string", 
                        "description": "The destination city name, e.g., London, Paris"
                    }
                },
                "required": ["city"]
            }
        }
    }]
    
    system_message = {"role": "system", "content": "You are a helpful assistant for Airline. Be Polite and accurate."}
    user_message = {"role": "user", "content": user_prompt}

    # First LLM call
    messages = [system_message] + history + [user_message]
    
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools
    )

    assistant_message = response.choices[0].message
    tool_calls = assistant_message.tool_calls
    
    if tool_calls:
        # Start by adding the assistant's request containing all tool calls
        messages.append(assistant_message)
        
        # FIX: Loop through EVERY tool call requested by the model
        for tool_call in tool_calls:
            tool_function = available_functions[tool_call.function.name]
            tool_args = json.loads(tool_call.function.arguments)
            
            # Execute tool function
            tool_output = tool_function(**tool_args)

            # Build individual tool response block
            tool_message = {
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": tool_call.function.name,
                "content": tool_output
            }
            # Append it right away so every tool ID gets a matching answer
            messages.append(tool_message)
    
        # 2nd LLM call after executing all tools
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )

    return response.choices[0].message.content


In [ ]:
print(chat("What is ticket price for paris, london, new york?", []))

In [16]:
chat("What is ticket price for Frankfurt?", [])


[Tool Executing] Looking up flight prices for: 'frankfurt'


'I’m sorry, but tickets for Frankfurt are currently not available. If you have any other destination in mind or need assistance with something else, please let me know!'

In [17]:
chat("How are you today?", [])

"I'm here and ready to help you! How can I assist you today?"

# wrap in gradio UI

In [18]:
import gradio as gr

In [19]:
assistant_ui = gr.ChatInterface(
    fn=chat
)

In [20]:
assistant_ui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.



[Tool Executing] Looking up flight prices for: 'new york'

[Tool Executing] Looking up flight prices for: 'frankfurt'

[Tool Executing] Looking up flight prices for: 'london'

[Tool Executing] Looking up flight prices for: 'munich'

[Tool Executing] Looking up flight prices for: 'paris'


In [13]:
gr.close_all()

Closing server running on port: 7860
